# Explore the XJTU-SY small subset with ISA-PHM

This notebook opens your existing local Wizard export, explores its studies and sensors, reads one raw measurement, and plots it. It does not modify the dataset. The CSV files are not included in this repository.

**Setup:** from the wrapper repository, run `git submodule update --init --recursive`, then `python -m pip install -e ".[notebooks]"` and `jupyter lab`. Select the kernel from that environment and run the cells in order. Change the path below if your dataset is elsewhere.

In [1]:
from pathlib import Path
import pandas as pd
from isa_phm import ISAWrapper
import numpy as np

dataset_root = Path(r'G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-Datasets-Small-Subset\Dataset')
isa_json = dataset_root / 'xjtu-sy-bearing-datasets-small-subset.isa-phm.json'
if not isa_json.is_file():
    raise FileNotFoundError(f'Edit dataset_root above to point to your extracted dataset: {isa_json}')
print('Dataset:', dataset_root)
print('ISA-PHM JSON:', isa_json)

g:\ISA\.venv\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Dataset: G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-Datasets-Small-Subset\Dataset
ISA-PHM JSON: G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-Datasets-Small-Subset\Dataset\xjtu-sy-bearing-datasets-small-subset.isa-phm.json


## Open the investigation

Strict ISA validation is disabled here so that this is an exploration example, not a conformance check. Use the specification validator separately for the checks it supports.

In [2]:
wrapper = ISAWrapper(isa_json, data_root=dataset_root, strict_validation=False)
wrapper.summary()

,title,identifier,experiment_type,n_studies,n_contacts,n_publications,description_short,source_path
0,XJTU-SY Bearing Datasets - Small,38d217b8-7deb-425d-82fb-eef1b80b0539,prognostics-experiment,2,2,0,A small subset of the XJTU-SY Bearing Dataset ...,G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-D...


In [3]:
pd.DataFrame([study.model_dump() for study in wrapper.list_studies()])

,study_id,title,n_assays,n_runs,n_factors
0,8d98377e-9b0d-4956-8f4d-5283385226f9,Bearing 1_1,2,123,5
1,3f8dd6ad-17df-484d-8342-54c0fac2ce8f,Bearing 1_2,2,161,5


## Choose a bearing and sensor

This export has two bearing studies. The first study has two sensor outputs, axial and radial. Assay indices start at 1; use the table to choose another sensor.

In [4]:
study = wrapper.study(2)
pd.DataFrame([assay.model_dump() for assay in study.list_assays()])

,assay_id,sensor_id,sensor_alias,technology_type,measurement_type,quantity_kind_identifier,physical_device_id,n_runs,n_raw_files,n_processed_files
0,se01,be71e503-a482-4655-9d5c-6217acca2c99,Accel_Axial,Accelerometer,Acceleration,http://qudt.org/vocab/quantitykind/Acceleration,None,161,161,0
1,se02,3638a9b3-a3f8-402e-ab22-1966897e2ec0,Accel_Radial,Accelerometer,Acceleration,http://qudt.org/vocab/quantitykind/Acceleration,None,161,161,0


In [5]:
assay = study.assay(1)
runs = assay.list_runs()
print(f'{len(runs)} measurements available for this sensor')
pd.DataFrame([item.model_dump() for item in runs[:5]])

161 measurements available for this sensor


,run_id,run_number,factor_values,raw_file_path,processed_file_path,n_measurement_params,n_processing_params
0,run_001,1,"{'Fault Type': 'Outer Race', 'RUL': '161 min',...",G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-D...,None,5,0
1,run_002,2,"{'Fault Type': 'Outer Race', 'RUL': '160 min',...",G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-D...,None,5,0
2,run_003,3,"{'Fault Type': 'Outer Race', 'RUL': '159 min',...",G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-D...,None,5,0
3,run_004,4,"{'Fault Type': 'Outer Race', 'RUL': '158 min',...",G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-D...,None,5,0
4,run_005,5,"{'Fault Type': 'Outer Race', 'RUL': '157 min',...",G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-D...,None,5,0


## Inspect a measurement

The wrapper's API calls these entries runs. The metadata describes the sampling rate, measured unit and operating conditions. Unit advisories are limited to known mismatches; no warning is not proof of scientific correctness.

In [6]:
run = assay.run(runs[0].run_id)
overview = run.overview()
print('Run:', run.run_id)
print('File:', overview.raw_file_path)
print('Factors:', run.factor_values())
print('Measurement parameters:')
display(pd.DataFrame(overview.measurement_params))
print('Capabilities and resolved metadata:')
display(run.analysis_capabilities().model_dump())
print('Quantity/unit advisory:', run.quantity_unit_advisory())

Run: run_001
File: G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-Datasets-Small-Subset\Dataset\Data\35Hz12kN\Bearing1_2\1_horizontal.csv
Factors: {'Fault Type': 'Outer Race', 'RUL': '161 min', 'Motor Speed': '2100 rev/min', 'Pressure Axial': '12 kN', 'Pressure Radial': '12 kN'}
Measurement parameters:


,parameter_name,value,unit,unit_identifier,value_identifier
0,Sampling Period,1,min,None,None
1,Sampling Points,32768,None,None,None
2,Sensor Orientation,90,°,None,None
3,Sampling Rate,25.6,kHz,None,None
4,Measured Unit,G,None,None,http://qudt.org/vocab/unit/G


Capabilities and resolved metadata:


{'run_id': 'run_001',
 'data_loading': {'available': True, 'reason': None},
 'time_domain_features': {'available': True, 'reason': None},
 'frequency_domain': {'available': True, 'reason': None},
 'physical_unit_interpretation': {'available': True, 'reason': None},
 'sampling_frequency': {'available': True,
  'value': 25600.0,
  'unit': 'Hz',
  'parameter_name': 'Sampling Rate',
  'source': 'measurement_parameter',
  'reason': None},
 'measured_unit': {'available': True,
  'value': 'G',
  'unit': 'G',
  'parameter_name': 'Measured Unit',
  'source': 'measurement_parameter',
  'reason': None}}

Quantity/unit advisory: QuantityUnitAdvisory(status='no_known_mismatch', warnings=(), reason=None)


## Load and plot the signal

This dataset contains raw files, so we explicitly request `file_type="raw"`. The wrapper converts the CSV time strings to seconds without resetting or rescaling them. Later measurements in this dataset start after zero; the timestamps do not by themselves prove synchronization.

In [7]:
df, load_info = run.load_dataframe_with_meta(file_type='raw')
print('Rows:', len(df))
print('Load details:', load_info.model_dump())
display(df.head())
display(df.describe())

Rows: 32768
Load details: {'assay_id': 'se01', 'run_id': 'run_001', 'requested_file_type': 'raw', 'resolved_file_type': 'raw', 'file_path': 'G:\\ISA\\Datasets\\Datasets-ISA\\XJTU-SY-Bearing-Datasets-Small-Subset\\Dataset\\Data\\35Hz12kN\\Bearing1_2\\1_horizontal.csv', 'from_cache': False, 'csv_engine': 'c', 'csv_sep': ',', 'csv_encoding': 'utf-8', 'csv_detection_source': 'common-separators', 'csv_bad_lines': 'error'}


,time,value
0,0.000000,0.157130
1,0.000039,-0.146127
2,0.000078,0.184369
3,0.000117,-0.078177
4,0.000156,-0.026917


,time,value
count,32768.000000,32768.000000
mean,0.639980,-0.004571
std,0.369510,0.619689
min,0.000000,-4.321063
25%,0.319990,-0.404721
50%,0.639981,-0.013477
75%,0.959971,0.388226
max,1.279961,3.902006


In [8]:
from bokeh.io import output_notebook, show
output_notebook()
show(run.plot_timeseries())

Loading BokehJS ...

## Compare a later measurement

To work with a zero-based axis for a particular analysis, create a separate copy. This does not alter the original CSV or the wrapper's returned values.

In [9]:
later_run = assay.run(runs[-1].run_id)
later_df = later_run.load_dataframe(file_type='raw')
analysis_df = later_df.copy()
analysis_df['time_from_start'] = analysis_df['time'] - analysis_df['time'].iloc[0]
print('Later measurement:', later_run.run_id)
display(analysis_df.head())

Later measurement: run_161


,time,value,time_from_start
0,9600.000000,3.949010,0.000000
1,9600.000039,5.847895,0.000039
2,9600.000078,5.216014,0.000078
3,9600.000117,11.157203,0.000117
4,9600.000156,9.792638,0.000156


## RMS and other features for one measurement

Use the wrapper to calculate features for this measurement. By default it prefers processed data if present, otherwise raw; the source is shown below. RMS includes any DC offset; no filtering or detrending is applied. Non-finite samples are excluded and counted. These features do not by themselves identify a fault.

In [10]:
import numpy as np
single_features = run.features()
display(pd.Series({key: value for key, value in single_features.items() if key != 'source'}, name='value'))
print('Source:', single_features['source']['resolved_file_type'], single_features['source']['file_path'])
print('Measured unit:', run.analysis_capabilities().measured_unit.value)

rms                 0.619697
max                 3.902006
mean               -0.004571
peak2peak           8.223069
kurtosis            0.932249
std                 0.619680
crest_factor        6.972866
skewness            0.080283
excluded_samples    0.000000
Name: value, dtype: float64

Source: raw G:\ISA\Datasets\Datasets-ISA\XJTU-SY-Bearing-Datasets-Small-Subset\Dataset\Data\35Hz12kN\Bearing1_2\1_horizontal.csv
Measured unit: G


## RMS across the bearing's measurements

The wrapper can calculate features across every measurement of the selected sensor. This reads all its CSV files and can take a little while. The horizontal axis is measurement order, not a claim of exact global time. A change in RMS alone does not establish degradation or a fault.

In [11]:
features = assay.lifecycle_features(file_type='raw', n_workers=1)
print('Measurement status counts:', features['status'].value_counts().to_dict())
display(features.loc[features['status'] != 'ok', ['run_id', 'file_path', 'error']])
display(features[['run_id', 'run_number', 'rms', 'peak2peak', 'kurtosis', 'crest_factor']].head())
selected = features.loc[features['run_id'] == run.run_id, 'rms']
print('Selected measurement RMS matches:', not selected.empty and np.isclose(selected.iloc[0], single_features['rms']))

Measurement status counts: {'ok': 161}


,run_id,file_path,error


,run_id,run_number,rms,peak2peak,kurtosis,crest_factor
0,run_001,1,0.619697,8.223069,0.932249,6.972866
1,run_002,2,0.611194,8.016682,1.093941,7.429372
2,run_003,3,0.625078,8.582914,1.162604,7.182885
3,run_004,4,0.601411,7.658839,1.005726,6.565154
4,run_005,5,0.593889,7.652998,0.874407,6.641121


Selected measurement RMS matches: True


In [12]:
from bokeh.plotting import figure

unit = run.analysis_capabilities().measured_unit.value or 'unit unavailable'
rms_plot = figure(title='RMS across measurements', x_axis_label='Measurement number', y_axis_label=f'RMS ({unit})', width=850, height=350)
rms_plot.line(features['run_number'].to_numpy(), features['rms'].to_numpy(), line_width=2)
show(rms_plot)

The wrapper also calculates kurtosis (Fisher/excess kurtosis) and crest factor. Interpretation depends on signal conditioning, sampling and operating conditions; avoid treating a single feature as a fault label.

## Frequency-domain view

The spectrum shows signal content by frequency. The wrapper reads the explicit sampling rate from the measurement metadata (25.6 kHz for this dataset). A peak alone does not identify a bearing fault; further analysis and machine context are needed.

In [13]:
show(assay.plot_frequency_domain(run_id=run.run_id, file_type='raw'))

## Prepare a feature table for modelling

Choose features explicitly with `feature_table()`. By default it stops if a selected feature or target is missing. Here we opt in to dropping incomplete measurements and inspect the accompanying report. The returned table includes source file information. This recalculates features for the selected sensor; no model is trained. RMS and peak-to-peak are examples, not universally recommended features.

In [14]:
ml_data, excluded = assay.feature_table(
    label_column='fv_RUL',
    features=['rms', 'peak2peak'],
    file_type='raw',
    n_workers=1,
    on_incomplete='drop',
)
print(f'Included: {len(ml_data)}; excluded: {len(excluded)}')
display(excluded)  # run IDs and reasons; empty when nothing was excluded
display(ml_data.head())

Included: 161; excluded: 0


,run_id,run_number,reasons


,run_id,run_number,rms,peak2peak,target
0,run_001,1,0.619697,8.223069,161 min
1,run_002,2,0.611194,8.016682,160 min
2,run_003,3,0.625078,8.582914,159 min
3,run_004,4,0.601411,7.658839,158 min
4,run_005,5,0.593889,7.652998,157 min


The RUL target in this export is text such as `123 min`; the wrapper does not convert it to a number. Before modelling, verify the meaning and units, then convert deliberately. Measurements from the same bearing are related: randomly splitting individual measurements can leak information between training and test sets. Consider splitting by bearing when evaluating a model.

## Try it yourself

Change `wrapper.study(1)` to `wrapper.study(2)`, or `study.assay(1)` to `study.assay(2)`, and rerun the following cells. You can also select another entry from `runs`. This example only demonstrates exploration; it is not a validation of the dataset or a diagnosis.